<a href="https://colab.research.google.com/github/Hanzet22/TKJ-Dumps/blob/main/Projek_Ana.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install requests

In [ ]:
# ================================================
#  🛠️ NetCare v1.0
#  Produk TKJ — Monitoring Jaringan Sederhana
#  Fungsi: Cek status perangkat, ping, port, internet
# ================================================

import subprocess
import platform
import socket
import time
import requests
from datetime import datetime
import ipaddress

print("""
   ╔═══════════════════════════════════════╗
   ║   🛠️  NetCare v1.0                   ║
   ║   Monitoring Jaringan Sederhana      ║
   ╚═══════════════════════════════════════╝
""")

# ================================================
# 1. CEK INTERNET
# ================================================
def cek_internet():
    try:
        requests.get('https://8.8.8.8', timeout=3)
        return True
    except:
        return False

# ================================================
# 2. PING HOST
# ================================================
def ping_host(host, count=4):
    param = "-n" if platform.system().lower() == "windows" else "-c"
    try:
        output = subprocess.run(["ping", param, str(count), host], capture_output=True, text=True)
        return output.stdout
    except:
        return None

# ================================================
# 3. SCAN PORT
# ================================================
def scan_port(host, port):
    try:
        sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
        sock.settimeout(1)
        result = sock.connect_ex((host, port))
        sock.close()
        return result == 0
    except:
        return False

# ================================================
# 4. GET IP & INFO
# ================================================
def get_ip_info():
    info = {}
    try:
        hostname = socket.gethostname()
        info['hostname'] = hostname
        info['ip_local'] = socket.gethostbyname(hostname)

        # IP Publik
        ip_pub = requests.get('https://api.ipify.org', timeout=3).text
        info['ip_publik'] = ip_pub
    except:
        info['ip_publik'] = "Gagal"
    return info

# ================================================
# MAIN MENU
# ================================================
while True:
    print("\n" + "="*50)
    print("🛠️  NETCARE — MENU UTAMA")
    print("="*50)
    print("1. Cek Status Internet")
    print("2. Ping Host")
    print("3. Scan Port (cepat: 1-1024)")
    print("4. Info IP & Hostname")
    print("5. Monitoring Server (ping + port)")
    print("6. Keluar")

    pilih = input("\nPilih menu (1-6): ").strip()

    if pilih == "1":
        print("\n📡 Mengecek koneksi internet...")
        if cek_internet():
            print("✅ Internet: TERHUBUNG")
        else:
            print("❌ Internet: TIDAK TERHUBUNG")

    elif pilih == "2":
        host = input("Masukkan IP/Domain: ").strip()
        print(f"\n⏳ Ping ke {host}...\n")
        hasil = ping_host(host)
        if hasil:
            print(hasil)
        else:
            print("❌ Gagal ping")

    elif pilih == "3":
        host = input("Masukkan IP/Domain: ").strip()
        print(f"\n🔍 Scan port 1-1024 di {host}...")
        terbuka = []
        for port in range(1, 1025):
            if scan_port(host, port):
                terbuka.append(port)
                try:
                    service = socket.getservbyport(port)
                except:
                    service = "unknown"
                print(f"   ✅ Port {port} ({service}) terbuka")
            if port % 100 == 0:
                print(f"   ⏳ Progress: {port}/1024", end='\r')
        print(f"\n✅ Selesai. Ditemukan {len(terbuka)} port terbuka.")
        if terbuka:
            print("   Port: ", ", ".join(map(str, terbuka[:10])))

    elif pilih == "4":
        info = get_ip_info()
        print("\n📡 INFO SISTEM:")
        print(f"   Hostname  : {info.get('hostname', 'N/A')}")
        print(f"   IP Lokal  : {info.get('ip_local', 'N/A')}")
        print(f"   IP Publik : {info.get('ip_publik', 'N/A')}")
        print(f"   OS        : {platform.system()} {platform.release()}")

    elif pilih == "5":
        host = input("Masukkan IP/Domain server: ").strip()
        print(f"\n🔄 Monitoring {host}...")
        print("Tekan Ctrl+C untuk berhenti.\n")
        try:
            while True:
                now = datetime.now().strftime("%H:%M:%S")
                status = "✅ UP" if ping_host(host, 1) else "❌ DOWN"
                print(f"[{now}] {host} — {status}")
                time.sleep(5)
        except KeyboardInterrupt:
            print("\n⏹️ Monitoring dihentikan.")

    elif pilih == "6":
        print("\n👋 Sampai jumpa! NetCare siap sedia.")
        break

    else:
        print("❌ Pilihan tidak valid. Coba lagi.")